# Session 2 · Day 2 - Center Recommendation on a Road Network

Day 1 produced a clean, PSGC-aligned center table with a coordinate-quality flag, and the
sibling notebook `S2_D2_01_analytical_questions_and_proximity_features` turned it into **proximity
features** on a straight-line distance matrix. That matrix ignores roads, rivers and one-way
streets. This notebook closes that gap for the **center-recommendation** problem: a learner picks a
qualification, and the task is to rank the centers that offer it by how far the learner must
actually **travel by road**.

The data is real and loads from the public bucket. The road network is built on the fly with
**OpenStreetMap** through **osmnx**, and the shortest paths come from **networkx**.

### What this notebook covers
1. Load the real NCR centers and attach the qualification each one is accredited to assess
2. Place a learner inside a real barangay polygon and keep the centers that serve that learner
3. Rank the centers by **straight-line** distance, the quick and rough answer
4. Build a small **drive network** around the learner and rank the centers by **road** distance
5. Compare the two rankings and explain when the road answer differs from the straight line
6. Engineer a **road-distance budget count** and a **k-nearest-by-road mean** *(exercise)*
7. Turn the ranking into a **top-k recommendation** and note how a national finder stays small

> **How to use this notebook**
> - ▶ **Run each cell in order** (`Shift+Enter`). Later cells depend on earlier ones.
> - 🌐 **Two cells reach the internet**: the OpenStreetMap download and the barangay-boundary
>   read. Each takes one to a few minutes, so read ahead while they run.
> - ✏️ **Exercise cells** have `### START CODE HERE ###` … `### END CODE HERE ###`. Write
>   code between those lines only.
> - ✅ After an exercise, run the **check cell**. Fix the code until it prints a pass.
> - ❓ **Question cells** take a short typed answer, checked against a value computed from the data.
> - 🐞 **Spot-the-flaw cells** show tempting but wrong code. Read, explain, then compare to the fix.
> - 🎛️ **Play-around cells** are open to tweak: change a learner or a qualification, re-run, compare.

### New to Python or pandas?

Anyone coming from another language can run `S2_commands_primer` first. It lays out the mental model in one page: a `DataFrame` is a table (a SQL table, an R `data.frame`, a spreadsheet), a `merge` is a `JOIN`, `NaN` is the missing value, and a chain of `.method().method()` reads left to right. The comments in the code cells point out each pandas idiom as it appears.

In [ ]:
# Run once per Colab session (skip if already installed locally).
!pip -q install osmnx networkx geopandas shapely pyproj folium mapclassify openpyxl

In [ ]:
import os, re, time
os.environ["CPL_VSIL_CURL_ALLOWED_EXTENSIONS"] = ".gpkg"   # let GDAL stream the remote GeoPackage

import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
import networkx as nx
import folium

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)
ox.settings.log_console = False

BASE = "https://storage.googleapis.com/tesda-datasets/data/"
print("osmnx", ox.__version__, "· networkx", nx.__version__, "· geopandas", gpd.__version__)

## The centers and what they offer

Two real tables combine into the supply side:

- **`tesda_coordinates.parquet`** is the geocoded center registry from Day 1. Each row carries a
  `coord_status`, and only `valid` pins are usable for routing.
- The **2026 Accredited Competency Assessment Center** database lists, for each center, the
  qualifications it is accredited to assess.

A center is a **candidate** for a learner only when it sits in NCR, has a `valid` pin, and is
accredited for the chosen qualification. The two tables are joined on a **normalized name**
(upper-cased, punctuation stripped), the same linking idea used in Day 1's entity-resolution
notebook.

In [ ]:
def normalize(s):
    """Upper-case, strip punctuation and collapse spaces, so two spellings of one name match."""
    s = re.sub(r"[^A-Z0-9 ]+", " ", str(s).upper())
    return re.sub(r"\s+", " ", s).strip()

centers = pd.read_parquet(BASE + "tesda_coordinates.parquet")
centers = centers[(centers["region"] == "NCR") & (centers["coord_status"] == "valid")][
    ["name", "latitude", "longitude", "coord_status", "region", "city_municipality"]].copy()
centers["key"] = centers["name"].map(normalize)

AC_URL = BASE + "2026%20Accredited%20Compentency%20Assessment%20Center%20based%20on%20July%2015%2C%202026%20database.csv"
accred = pd.read_csv(AC_URL, dtype=str)
accred = accred[accred["Region Name"].str.contains("NCR", na=False)].copy()
accred["key"] = accred["Assessment Center"].map(normalize)
print(f"{len(centers):,} NCR centers with a valid pin · {len(accred):,} NCR accreditation rows")

Several hundred NCR centers carry a usable pin, across roughly 1,500 NCR
accreditation rows (one center can appear many times, once per qualification).

Pick a qualification and keep the centers accredited to assess it. **Bread and Pastry Production
NC II** is the most-accredited qualification in the country, so it makes a dense, interesting map.

In [ ]:
QUALIFICATION = "Bread and Pastry Production NC II"

offer_keys = set(accred.loc[accred["Qualification"] == QUALIFICATION, "key"])
candidates = (centers[centers["key"].isin(offer_keys)]   # .isin = SQL IN: keep centers whose key is in the set
              .drop_duplicates("key")
              .reset_index(drop=True)
              .copy())
print(f"{len(offer_keys):,} NCR centers are accredited for '{QUALIFICATION}'")
print(f"{len(candidates):,} of them also have a valid pin to route to")
candidates[["name", "city_municipality", "latitude", "longitude"]].head()

The candidate count is only a floor. A center whose name
does not match between the two tables, or whose pin failed Day 1's coordinate check, drops out
silently. A production finder would surface those as "location unconfirmed".

## Choose a learner

A real registration gives the learner's **barangay**. The barangay polygons live in
`consolidated_geodata_matched.gpkg` (PSGC-coded, EPSG:4326). The file is large, so GDAL streams
only the features inside a small bounding box around Quezon City.
NCR carries the admin-1 code **`PH13`**, and each city carries an admin-3 code (Quezon City is
`PH1307404`), which disambiguates barangay names that repeat across cities.

The learner's location is the barangay's **representative point**, a point guaranteed to fall
inside the polygon.

In [ ]:
GPKG = "/vsicurl/" + BASE + "consolidated_geodata_matched.gpkg"
LEARNER_BARANGAY = "Pinyahan"
LEARNER_CITY = "Quezon City"
LEARNER_CITY_PCODE = "PH1307404"   # Quezon City, from the gpkg admin-3 code

t0 = time.time()
qc_box = (121.00, 14.60, 121.09, 14.69)                 # lon_min, lat_min, lon_max, lat_max
brgys = gpd.read_file(GPKG, bbox=qc_box)
brgys = brgys[brgys["adm1_pcode"] == "PH13"]            # NCR only
row = brgys[(brgys["adm3_pcode"] == LEARNER_CITY_PCODE)
            & (brgys["adm4_en_psgc"].str.lower() == LEARNER_BARANGAY.lower())].iloc[0]

learner_pt = row.geometry.representative_point()
learner_lat, learner_lon = learner_pt.y, learner_pt.x
learner_label = f"{row['adm4_en_psgc']}, {LEARNER_CITY}"
print(f"Learner: {learner_label}  ({learner_lat:.5f}, {learner_lon:.5f})  ·  boundary read in {time.time() - t0:.0f} s")

One line names the barangay and city, with coordinates near latitude 14.64
and longitude 121.05. The boundary read touches the network, so it takes a few seconds to a minute.

## Distance as the crow flies

The **haversine** formula gives the straight-line (great-circle) distance between two
latitude/longitude points. It needs no road data, so it is instant even for thousands of centers,
which is exactly why it is the right **first** pass.

Straight-line distance and network distance trade off speed against realism.

- *Straight-line* (haversine) is free and instant, but it ignores every barrier a trip must cross:
  rivers, limited-access highways and one-way streets. Its advantage is speed. Its risk is that two
  centers equally close on the map can be very different trips.
- *Network* distance follows the road graph, so it reflects the trip a learner actually makes. Its
  advantage is realism. Its cost is a graph to build and a shortest-path search to run.

The sound pattern is **straight line first to narrow the field, road second to rank what is left**,
which is the pre-filter idea this notebook ends on.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Works on single numbers or whole columns."""
    R = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(np.subtract(lon2, lon1)) / 2) ** 2)
    return 2 * R * np.arcsin(np.sqrt(a))

candidates["straight_km"] = haversine_km(learner_lat, learner_lon,
                                         candidates["latitude"].values, candidates["longitude"].values)
candidates["rank_straight"] = candidates["straight_km"].rank(method="first").astype(int)   # rank(method="first") breaks ties by order of appearance
candidates.sort_values("straight_km")[["name", "city_municipality", "straight_km", "rank_straight"]].head(5).round(2)

The five centers nearest the learner on a straight line appear closest first. It is a
ranking that no learner can actually travel: rivers, rail lines and one-way streets all sit between
the pin and these centers. The next section measures the trip by road.

## Rank by road with one shortest-path search

The road ranking takes four moves:

1. **Build** a drivable network around the learner with osmnx. The radius is kept small (about
   8 km) so the download and the search finish in a couple of minutes. A learner in central
   NCR has many accredited centers well inside that radius.
2. **Snap** the learner and every nearby candidate to its closest road node.
3. Run **one** `networkx` shortest-path (Dijkstra) search from the learner's node. One search
   returns the road distance to *every* node at once, so 5 candidates or 500 cost the same.
4. **Look up** each candidate's node in that result.

The download reads live OpenStreetMap data, so it needs a working connection and can take one to
three minutes. A good moment to read ahead.

In [ ]:
GRAPH_DIST_M = 8000          # radius of the drive network around the learner (metres)

t0 = time.time()
G_drive = ox.graph.graph_from_point((learner_lat, learner_lon),
                                    dist=GRAPH_DIST_M, network_type="drive", simplify=True)
G_drive = ox.routing.add_edge_speeds(G_drive)          # km/h per edge, from OSM tags or defaults
G_drive = ox.routing.add_edge_travel_times(G_drive)    # seconds per edge, from length and speed
print(f"Drive network: {len(G_drive):,} nodes, {G_drive.number_of_edges():,} edges "
      f"(built in {time.time() - t0:.0f} s)")

Only centers inside the network can be routed to, so the field is pre-filtered to candidates
within the graph radius before snapping. This is the straight-line pre-filter in miniature: the
cheap haversine decides who is worth routing.

In [ ]:
near = candidates[candidates["straight_km"] <= GRAPH_DIST_M / 1000].copy()

learner_node = ox.distance.nearest_nodes(G_drive, X=learner_lon, Y=learner_lat)   # snap the point to the closest graph node
near["node"] = ox.distance.nearest_nodes(G_drive, X=near["longitude"].values, Y=near["latitude"].values)

node_y = {n: d["y"] for n, d in G_drive.nodes(data=True)}   # dict comprehension: node id -> its latitude
node_x = {n: d["x"] for n, d in G_drive.nodes(data=True)}
near["snap_m"] = haversine_km(near["latitude"], near["longitude"],
                              near["node"].map(node_y), near["node"].map(node_x)) * 1000   # .map(dict) looks each node's coordinate up
print(f"{len(near)} candidates within the {GRAPH_DIST_M/1000:.0f} km radius · "
      f"median snap {near['snap_m'].median():.0f} m · "
      f"{(near['snap_m'] > 250).sum()} center(s) more than 250 m from a road")

A snap of a few dozen metres is healthy. A center more than 250 m from any drivable road deserves a
second look: a large campus, a pin dropped on a building roof, or a wrong pin.

Now the search itself. `single_source_dijkstra_path_length` returns a dictionary
`{node: distance from the learner}`. It runs twice, once weighted by **length** (metres) and once
by **travel_time** (seconds, from osmnx's assumed free-flow speeds).

In [ ]:
road_m = nx.single_source_dijkstra_path_length(G_drive, learner_node, weight="length")   # one search returns shortest distance from the learner to every node
road_s = nx.single_source_dijkstra_path_length(G_drive, learner_node, weight="travel_time")

near["drive_km"] = near["node"].map(road_m) / 1000   # look each candidate's node up in that result dict
near["drive_min"] = near["node"].map(road_s) / 60
near = near.dropna(subset=["drive_km"]).copy()          # drop any candidate the search could not reach
near["rank_drive"] = near["drive_km"].rank(method="first").astype(int)
print(f"Reached {len(near)} candidates by road")

TOP_N = 5
near.sort_values("drive_km")[["name", "city_municipality", "drive_km", "drive_min",
                              "straight_km", "rank_straight"]].head(TOP_N).round(2)

`drive_min` assumes **no traffic** (free-flow speeds), so it is a lower bound. Compare each top center's `rank_straight` with its position in this road-ranked table: a
center that is #1 by road but #3 by straight line is exactly the case the crow-flies answer would
have gotten wrong.

## When the road disagrees with the straight line

The **detour factor** is `drive_km / straight_km`: how much longer the road is than the straight
line. Day 1 showed it is not a constant. A factor near 1 means the road runs almost straight to the
center. A large factor means something, a river, a rail line, a limited-access road, forces a
long way around.

In [ ]:
compare = near.sort_values("drive_km").head(10).copy()
compare["detour"] = compare["drive_km"] / compare["straight_km"]
compare[["name", "straight_km", "rank_straight", "drive_km", "rank_drive", "detour"]].round(2)

Where `rank_straight` and `rank_drive` disagree, the straight line has mis-ranked a center. The
center with the **largest detour factor** is the one to find on the map: something physical sits
between it and the learner.

### 🐞 Spot the flaw

A teammate proposes this one-liner for the recommendation, arguing it is "basically the same and
far faster":

```python
# Recommend the single closest center, measured as the crow flies.
recommended = candidates.sort_values("straight_km").iloc[0]
```

It runs, and for a learner on an open street grid it often even agrees with the road answer. Read
the next cell, which puts the straight-line #1 and the road #1 side by side, then explain in one
sentence **when** this shortcut sends a learner to the wrong center.

In [ ]:
straight_1 = near.sort_values("straight_km").iloc[0]
road_1 = near.sort_values("drive_km").iloc[0]
print("Straight-line #1:", straight_1["name"],
      f"({straight_1['straight_km']:.2f} km straight, {straight_1['drive_km']:.2f} km by road)")
print("Road #1:        ", road_1["name"],
      f"({road_1['straight_km']:.2f} km straight, {road_1['drive_km']:.2f} km by road)")
print("Same center?", straight_1["name"] == road_1["name"])
worst = near.assign(detour=near["drive_km"] / near["straight_km"]).sort_values("detour").iloc[-1]
print(f"Largest detour: {worst['name']} at {worst['straight_km']:.2f} km straight but "
      f"{worst['drive_km']:.2f} km by road (x{worst['drive_km']/worst['straight_km']:.2f})")

**The flaw:** straight-line distance ignores what lies between two points. When a river, a rail
line or a limited-access highway separates the learner from a close-looking center, the only
crossing can be far away, so the real trip is much longer than the map suggests, and a center that
looked second or third becomes the nearest one to actually reach. Ranking by `straight_km` across
such a barrier recommends a center the learner cannot easily get to. The fix is to rank by
`drive_km`, which is what the search above computes.

## ✏️ Exercise - two road-distance features

A recommendation needs more than the single nearest center. Engineer two features on the road
distances already in `near` and store them in the given variables:

- **`within_budget`**: the **number of candidates reachable within a 3 km road trip** (`drive_km`
  below 3).
- **`k3_road_mean`**: the **mean road distance to the 3 nearest centers by road** (in km).

Complete the body between the markers. *Hint:* `near["drive_km"]` is a column of road distances in
km. `np.sort(...)` returns the values in ascending order, and `(series < 3).sum()` counts how many
are below a threshold.

In [ ]:
### START CODE HERE ###   (about 2 lines)
within_budget = None     # count of candidates with drive_km < 3
k3_road_mean = None       # mean of the 3 smallest drive_km values, in km
### END CODE HERE ###
print("Candidates within a 3 km road trip:", within_budget)
print(f"Mean road distance to the 3 nearest centers: {k3_road_mean:.2f} km")

## ✅ Check the result
Run this. It recomputes both features straight from `near` and compares.

In [ ]:
budget_ref = int((near["drive_km"] < 3).sum())
k3_ref = float(np.sort(near["drive_km"].values)[:3].mean())
try:
    assert within_budget is not None and k3_road_mean is not None, \
        "One of the variables is still None. Did you fill in the code?"
    assert int(within_budget) == budget_ref, \
        f"within_budget should be {budget_ref}, got {within_budget}"
    assert np.isclose(k3_road_mean, k3_ref), \
        f"k3_road_mean should be {k3_ref:.3f}, got {k3_road_mean}"
    print(f"✅ Passed! {budget_ref} center(s) within a 3 km road trip; "
          f"3-nearest road mean = {k3_ref:.2f} km.")
except AssertionError as e:
    print("❌ Not yet:", e)

### ❓ Question

Among the routed candidates in `near`, how many have a **detour factor above 1.3**
(`drive_km / straight_km > 1.3`)? These are the centers the straight-line ranking is most likely to
misplace. Fill in the integer count.

In [ ]:
q_answer = None   # <- answer here: an integer count

In [ ]:
ref = int(((near["drive_km"] / near["straight_km"]) > 1.3).sum())
try:
    assert q_answer is not None, "q_answer is still None. Fill in your answer."
    assert int(q_answer) == ref, f"expected {ref}, got {q_answer}"
    print(f"✅ Passed! {ref} candidate(s) have a detour factor above 1.3.")
except AssertionError as e:
    print("❌ Not yet:", e)

## The top-k recommendation

A finder is a **filter followed by a ranking**. The filter kept centers with the right
qualification, a valid pin and a reachable road node. The ranking orders them by road distance.
The recommendation is simply the top of that list, with the straight-line and road numbers side by
side so the learner can see the trade-off.

In [ ]:
recommendation = (near.sort_values("drive_km")
                  .head(5)
                  .assign(learner=learner_label, qualification=QUALIFICATION)
                  [["learner", "qualification", "name", "city_municipality",
                    "drive_km", "drive_min", "straight_km", "rank_straight"]]
                  .reset_index(drop=True)
                  .round(2))
recommendation

A map makes the recommendation concrete: the learner, the top centers, and the actual road path to
the road-nearest one.

In [ ]:
best = near.sort_values("drive_km").iloc[0]
m = folium.Map(location=[learner_lat, learner_lon], zoom_start=14, tiles="CartoDB positron")
route = nx.shortest_path(G_drive, learner_node, best["node"], weight="length")
if len(route) > 1:
    pts = [(G_drive.nodes[n]["y"], G_drive.nodes[n]["x"]) for n in route]
    folium.PolyLine(pts, color="#30638e", weight=5, opacity=0.8, tooltip="road path to #1").add_to(m)
for _, r in near.sort_values("drive_km").head(5).iterrows():
    folium.CircleMarker([r["latitude"], r["longitude"]], radius=7, color="#edae49",
                        fill=True, fill_opacity=0.9,
                        tooltip=f"{r['name']} · road {r['drive_km']:.1f} km · straight {r['straight_km']:.1f} km"
                        ).add_to(m)
folium.Marker([learner_lat, learner_lon], tooltip=f"Learner: {learner_label}",
              icon=folium.Icon(color="green", icon="user")).add_to(m)
m

## 🎛️ Play around - a different learner or qualification

The recommendation hides two choices: **who** the learner is and **what** they want. Change either
in the cell below, then re-run from **Choosing a learner** (the qualification can be changed from
**The centers and what they offer**). A new learner or a sparser qualification can empty the
walkable field, move the #1 center, or raise the detour factors. The cell prints valid options to
pick from.

In [ ]:
# Qualifications with the most NCR accreditations (any of these is a valid QUALIFICATION):
print("Try these qualifications:")
print(accred["Qualification"].value_counts().head(8).to_string())

# Barangays available in the Quezon City box already read above (any is a valid LEARNER_BARANGAY):
qc_names = sorted(brgys.loc[brgys["adm3_pcode"] == LEARNER_CITY_PCODE, "adm4_en_psgc"].dropna().unique())
print("\nSome Quezon City barangays to try:", ", ".join(qc_names[:12]))
print("\nCurrent choice:", LEARNER_BARANGAY, "·", QUALIFICATION)

## Scale up without downloading the country

One small network around one learner is cheap. A national, always-on finder needs two changes, and
neither means routing every learner against every center.

- **Pre-build the network, do not download per learner.** Day 1 clipped the whole of NCR into
  `ncr_clip.osm.pbf`. A city-wide or region-wide drive graph built once from that file and saved
  (for example as GraphML) is loaded in seconds, so the live OpenStreetMap download above is a
  teaching convenience.
- **Filter cheap, route dear.** Rank all centers by the free haversine distance, then run the road
  search only on the **k** nearest. One Dijkstra search per learner already returns every node at
  once within a city graph. Across the whole country the standard engine is **OSRM** (see
  `S2_D2_00_osrm_routing`), where each learner-to-center pair is one request, so the top-k
  pre-filter is what keeps the request count small. Pick **k** by measuring how often the true
  road-nearest center falls outside the straight-line top k, and state that k in the method.

## Recap
- A finder is a **filter then a ranking**: keep centers with the right qualification, a valid pin
  and a reachable road node, then order them by road distance.
- **Straight-line** distance is the fast first pass. **Road** distance from one Dijkstra search is
  the ranking to trust, and the two disagree exactly where a barrier lies between learner and center.
- The **detour factor** flags those disagreements, and desk-checking the #1 route against an
  independent map keeps a wrong pin from reaching a learner.
- A national finder stays small with a **straight-line pre-filter** and a pre-built network or OSRM.

**Next:** `S2_D2_02_accessibility_and_2sfca`: from one learner's ranking to whole-area accessibility
and the 2SFCA method.

## References & further reading

Documentation and background for the tools and methods in this notebook.

- [OSMnx, documentation](https://osmnx.readthedocs.io/)
- [OSMnx, get a graph from a point](https://osmnx.readthedocs.io/en/stable/user-reference.html#osmnx.graph.graph_from_point)
- [NetworkX, shortest paths (Dijkstra)](https://networkx.org/documentation/stable/reference/algorithms/shortest_paths.html)
- [OpenStreetMap, about](https://www.openstreetmap.org/about)
- [GeoPandas, reading and writing files](https://geopandas.org/en/stable/docs/user_guide/io.html)
- [Project OSRM (national-scale routing)](https://project-osrm.org/)
- [Haversine formula (overview)](https://en.wikipedia.org/wiki/Haversine_formula)